# Cross-Speed Generalisation Study — New May 2026 Samples

Supervised by Prof. Elmar Rückert — Montanuniversität Leoben

## Experiment design

Two symmetric experiments using only the new May 2026 data:

| Experiment | Train | Test |
|---|---|---|
| **A** | Fractions of new **5.10 Hz** data | New **1.83 Hz** data (full, fixed) |
| **B** | Fractions of new **1.83 Hz** data | New **5.10 Hz** data (full, fixed) |

**Scientific question:** can a ResNet-18 trained at one belt speed generalise
to the other speed, and how much training data does it need?

Belt speed affects how long the laser dwells on each rock grain —
faster belt = shorter integration time = noisier spectra.
If the model generalises well across speeds, it means the spectral
shape (peak positions, hump profile) is speed-invariant,
only the signal-to-noise ratio changes.

- **Fractions**: 20%, 30%, 40%, 50%, 60%, 70%, 80%, 100%
- **Seeds**: 3 per fraction → 24 runs per experiment → 48 total
- **Training images per class at 100%**: 400
- **Test set**: fixed 400 images per class from the other speed

**Output folder:** `results_cross_speed/`

In [ ]:
import os, random, warnings, csv as _csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
print('Seed set.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════

DATA_ROOT = os.path.expanduser('~/20260511_New_Granite-Limestone-Sandstone_224')

RESULTS_DIR  = 'results_cross_speed'
DIR_WEIGHTS  = os.path.join(RESULTS_DIR, 'weights')
DIR_PLOTS    = os.path.join(RESULTS_DIR, 'plots')
DIR_CSV      = os.path.join(RESULTS_DIR, 'csv')
for d in [RESULTS_DIR, DIR_WEIGHTS, DIR_PLOTS, DIR_CSV]:
    os.makedirs(d, exist_ok=True)

CLASS_NAMES  = ['S10Granite', 'Holstein_Sandstone', 'Leitendorf_Limestone']
SHORT_NAMES  = ['Granite', 'Sandstone', 'Limestone']
CLASS_COLORS = ['#185FA5', '#3B6D11', '#854F0B']
VALID_EXT    = ('.jpg', '.jpeg', '.bmp', '.png')

FRACTIONS = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 1.00]
SEEDS     = [7, 42, 123]

EPOCHS     = 20
LR         = 1e-4
WD         = 1e-4
BATCH_SIZE = 32   # small — dataset is 400 images per class at 100%
VAL_SPLIT  = 0.20

_saved_files = []
def save_fig(fig, folder, filename, description, dpi=150):
    path = os.path.join(folder, filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    _saved_files.append((path, description))
    print(f'[SAVED] {path}')

print('Config ready.')
print(f'  DATA_ROOT = {DATA_ROOT}')
print(f'  Fractions : {[int(f*100) for f in FRACTIONS]}%')
print(f'  Seeds     : {SEEDS}')
print(f'  Total runs: {len(FRACTIONS) * len(SEEDS) * 2} (2 experiments × {len(FRACTIONS)} fractions × {len(SEEDS)} seeds)')

In [ ]:
# LOAD ALL 6 FOLDERS
# Returns dict: speed_tag -> {class_idx: [paths]}

FOLDERS = {
    '5-10Hz': {
        0: os.path.join(DATA_ROOT, 'New_S10Granite_5-10Hz'),
        1: os.path.join(DATA_ROOT, 'New_Holstein_Sandstone_5-10Hz'),
        2: os.path.join(DATA_ROOT, 'New_Leitendorf_Limestone_5-10Hz'),
    },
    '1-83Hz': {
        0: os.path.join(DATA_ROOT, 'New_S10Granite_1-83Hz'),
        1: os.path.join(DATA_ROOT, 'New_Holstein_Sandstone_1-83Hz'),
        2: os.path.join(DATA_ROOT, 'New_Leitendorf_Limestone_1-83Hz'),
    },
}

all_paths = {}   # speed_tag -> {class_idx: [paths]}
for speed, class_folders in FOLDERS.items():
    all_paths[speed] = {}
    print(f'\n{speed}:')
    for ci, folder in class_folders.items():
        paths = sorted([str(p) for p in Path(folder).iterdir()
                        if p.suffix.lower() in VALID_EXT])
        all_paths[speed][ci] = paths
        print(f'  {SHORT_NAMES[ci]:12s}: {len(paths)} images  ({folder})')

print('\nData loaded OK.')

In [ ]:
# DATASET CLASS + TRANSFORMS

train_tf = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_tf = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class RockDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = labels
        self.transform = transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        return self.transform(img), self.labels[idx]


print('Dataset class and transforms ready.')

In [ ]:
# MODEL + TRAIN / EVALUATE FUNCTIONS

def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, len(CLASS_NAMES))
    return m.to(device)


def train_one_run(train_speed, fraction, seed):
    """Train ResNet-18 on `fraction` of `train_speed` data. Returns trained model."""
    weight_path = os.path.join(
        DIR_WEIGHTS,
        f'cross_speed_train{train_speed}_frac{int(fraction*100):03d}_seed{seed}.pth')

    model = build_model()
    if os.path.exists(weight_path):
        model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
        model.eval()
        return model

    seed_everything(seed)

    # Build training set from fraction of train_speed data
    tr_paths, tr_labels = [], []
    for ci, paths in all_paths[train_speed].items():
        n = max(1, int(len(paths) * fraction))
        random.seed(seed)
        sampled = random.sample(paths, n)
        tr_paths.extend(sampled)
        tr_labels.extend([ci] * n)

    # Train/val split within the training speed
    tr_p, va_p, tr_l, va_l = train_test_split(
        tr_paths, tr_labels, test_size=VAL_SPLIT, stratify=tr_labels, random_state=seed)

    tr_ldr = DataLoader(RockDataset(tr_p, tr_l, train_tf),
                        BATCH_SIZE, shuffle=True,  num_workers=0)
    va_ldr = DataLoader(RockDataset(va_p, va_l, test_tf),
                        BATCH_SIZE, shuffle=False, num_workers=0)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()
    use_amp   = device.type == 'cuda'
    scaler    = torch.amp.GradScaler('cuda') if use_amp else None

    best_val = 0.0
    for epoch in range(EPOCHS):
        model.train()
        for Xb, yb in tr_ldr:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            if use_amp:
                with torch.amp.autocast('cuda'):
                    out = model(Xb); loss = criterion(out, yb)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                out = model(Xb); loss = criterion(out, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        scheduler.step()

        model.eval()
        correct = total = 0
        with torch.no_grad():
            for Xb, yb in va_ldr:
                correct += (model(Xb.to(device)).argmax(1) == yb.to(device)).sum().item()
                total   += len(yb)
        val_acc = correct / total * 100
        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), weight_path)

    model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model.eval()
    return model


def evaluate_on_speed(model, test_speed):
    """Evaluate model on ALL images from test_speed. Returns per-class accuracy."""
    results = {}
    for ci, paths in all_paths[test_speed].items():
        ds  = RockDataset(paths, [ci]*len(paths), test_tf)
        ldr = DataLoader(ds, BATCH_SIZE, shuffle=False, num_workers=0)
        correct = total = 0
        with torch.no_grad():
            for Xb, yb in ldr:
                correct += (model(Xb.to(device)).argmax(1) == yb.to(device)).sum().item()
                total   += len(yb)
        results[ci] = correct / total * 100
    results['overall'] = np.mean(list(results.values()))
    return results


print('Model and training functions ready.')

In [ ]:
# MAIN EXPERIMENT LOOP
# experiment_results[exp][fraction][seed] = {ci: acc, 'overall': acc}

EXPERIMENTS = {
    'A': {'train_speed': '5-10Hz', 'test_speed': '1-83Hz',
          'label': 'Exp A: Train 5.10Hz → Test 1.83Hz'},
    'B': {'train_speed': '1-83Hz', 'test_speed': '5-10Hz',
          'label': 'Exp B: Train 1.83Hz → Test 5.10Hz'},
}

experiment_results = {exp: {f: {} for f in FRACTIONS} for exp in EXPERIMENTS}

total_runs = len(EXPERIMENTS) * len(FRACTIONS) * len(SEEDS)
run_num    = 0

for exp_id, exp_cfg in EXPERIMENTS.items():
    train_speed = exp_cfg['train_speed']
    test_speed  = exp_cfg['test_speed']
    print(f'\n{"="*60}')
    print(f'  {exp_cfg["label"]}')
    print(f'{"="*60}')

    for fraction in FRACTIONS:
        for seed in SEEDS:
            run_num += 1
            pct = int(fraction * 100)
            n_train = int(400 * fraction)  # approx per class
            print(f'  [{run_num:>2}/{total_runs}] Exp {exp_id}  '
                  f'frac={pct}%  seed={seed}  (~{n_train} imgs/class)')

            model = train_one_run(train_speed, fraction, seed)
            res   = evaluate_on_speed(model, test_speed)
            experiment_results[exp_id][fraction][seed] = res

            print(f'    Granite={res[0]:.1f}%  '
                  f'Sandstone={res[1]:.1f}%  '
                  f'Limestone={res[2]:.1f}%  '
                  f'Overall={res["overall"]:.1f}%')

            del model
            torch.cuda.empty_cache()

print('\nAll runs complete.')

In [ ]:
# AGGREGATE — mean ± std across seeds

# agg[exp_id][fraction][metric] = {'mean': float, 'std': float}
agg = {}
for exp_id in EXPERIMENTS:
    agg[exp_id] = {}
    for fraction in FRACTIONS:
        agg[exp_id][fraction] = {}
        for metric in [0, 1, 2, 'overall']:
            vals = [
                experiment_results[exp_id][fraction][seed][metric]
                for seed in SEEDS
                if seed in experiment_results[exp_id][fraction]
            ]
            if vals:
                agg[exp_id][fraction][metric] = {
                    'mean': float(np.mean(vals)),
                    'std':  float(np.std(vals)),
                }

# Save to CSV
csv_path = os.path.join(DIR_CSV, 'cross_speed_results.csv')
with open(csv_path, 'w', newline='') as f:
    w = _csv.writer(f)
    w.writerow(['experiment', 'train_speed', 'test_speed', 'fraction_pct',
                'granite_mean', 'sandstone_mean', 'limestone_mean', 'overall_mean',
                'granite_std',  'sandstone_std',  'limestone_std',  'overall_std'])
    for exp_id, exp_cfg in EXPERIMENTS.items():
        for fraction in FRACTIONS:
            if fraction not in agg[exp_id]: continue
            d = agg[exp_id][fraction]
            w.writerow([
                exp_id, exp_cfg['train_speed'], exp_cfg['test_speed'],
                int(fraction * 100),
                round(d[0]['mean'], 2),  round(d[1]['mean'], 2),
                round(d[2]['mean'], 2),  round(d['overall']['mean'], 2),
                round(d[0]['std'], 2),   round(d[1]['std'], 2),
                round(d[2]['std'], 2),   round(d['overall']['std'], 2),
            ])

_saved_files.append((csv_path, 'Cross-speed results CSV'))
print(f'[SAVED] {csv_path}')

In [ ]:
# RES-01  Overall accuracy vs training fraction — both experiments

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
fig.suptitle(
    'RES-01 | Cross-Speed Generalisation — Overall Accuracy vs Training Fraction\n'
    'Shading = ±1 std across seeds  |  Dashed = 90% reference line',
    fontsize=11, fontweight='bold')

for ax, (exp_id, exp_cfg) in zip(axes, EXPERIMENTS.items()):
    fracs = [f for f in FRACTIONS if f in agg[exp_id]]
    means = [agg[exp_id][f]['overall']['mean'] for f in fracs]
    stds  = [agg[exp_id][f]['overall']['std']  for f in fracs]
    pcts  = [int(f * 100) for f in fracs]

    ax.plot(pcts, means, 'o-', color='#185FA5', lw=2, ms=7)
    ax.fill_between(pcts,
                    [m-s for m, s in zip(means, stds)],
                    [m+s for m, s in zip(means, stds)],
                    alpha=0.2, color='#185FA5')
    ax.axhline(90, color='green', ls='--', lw=1.2, alpha=0.7, label='90% line')
    ax.set_title(exp_cfg['label'], fontsize=10, fontweight='bold')
    ax.set_xlabel('Training fraction (%)')
    ax.set_ylabel('Overall accuracy (%)')
    ax.set_xticks(pcts)
    ax.set_ylim(50, 105)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, DIR_PLOTS, 'RES-01_overall_accuracy_vs_fraction.png',
    'Overall accuracy vs training fraction for both experiments')
plt.show()

In [ ]:
# RES-02  Per-class accuracy vs fraction — both experiments side by side

fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey=True)
fig.suptitle(
    'RES-02 | Per-Class Accuracy vs Training Fraction\n'
    'Top = Exp A (train 5.10Hz, test 1.83Hz)  |  '
    'Bottom = Exp B (train 1.83Hz, test 5.10Hz)\n'
    'Shading = ±1 std across seeds',
    fontsize=11, fontweight='bold')

for row_idx, (exp_id, exp_cfg) in enumerate(EXPERIMENTS.items()):
    for col_idx, (ci, short, color) in enumerate(
            zip([0, 1, 2], SHORT_NAMES, CLASS_COLORS)):
        ax    = axes[row_idx, col_idx]
        fracs = [f for f in FRACTIONS if f in agg[exp_id] and ci in agg[exp_id][f]]
        means = [agg[exp_id][f][ci]['mean'] for f in fracs]
        stds  = [agg[exp_id][f][ci]['std']  for f in fracs]
        pcts  = [int(f * 100) for f in fracs]

        ax.plot(pcts, means, 'o-', color=color, lw=2, ms=7)
        ax.fill_between(pcts,
                        [m-s for m, s in zip(means, stds)],
                        [m+s for m, s in zip(means, stds)],
                        alpha=0.2, color=color)
        ax.axhline(90, color='gray', ls='--', lw=1.0, alpha=0.6)
        ax.set_title(f'{short}\n{exp_cfg["label"]}', fontsize=9)
        ax.set_xlabel('Training fraction (%)')
        ax.set_ylabel('Accuracy (%)')
        ax.set_xticks(pcts)
        ax.set_ylim(40, 105)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, DIR_PLOTS, 'RES-02_per_class_accuracy.png',
    'Per-class accuracy vs fraction for both experiments')
plt.show()

In [ ]:
# RES-03  Accuracy heatmap — class × fraction, both experiments

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle(
    'RES-03 | Cross-Speed Accuracy Heatmap (mean across seeds)\n'
    'Rows = class  |  Cols = training fraction  |  Green = good',
    fontsize=11, fontweight='bold')

for ax, (exp_id, exp_cfg) in zip(axes, EXPERIMENTS.items()):
    fracs = [f for f in FRACTIONS if f in agg[exp_id]]
    mat   = np.zeros((3, len(fracs)))
    for fi, f in enumerate(fracs):
        for ci in range(3):
            if ci in agg[exp_id][f]:
                mat[ci, fi] = agg[exp_id][f][ci]['mean']

    im = ax.imshow(mat, vmin=40, vmax=100, cmap='RdYlGn', aspect='auto')
    ax.set_yticks(range(3)); ax.set_yticklabels(SHORT_NAMES)
    ax.set_xticks(range(len(fracs)))
    ax.set_xticklabels([f'{int(f*100)}%' for f in fracs])
    ax.set_xlabel('Training fraction')
    ax.set_title(exp_cfg['label'], fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Accuracy (%)')
    for ci in range(3):
        for fi in range(len(fracs)):
            ax.text(fi, ci, f'{mat[ci,fi]:.0f}%',
                    ha='center', va='center', fontsize=9, fontweight='bold',
                    color='white' if mat[ci,fi] < 65 else 'black')

plt.tight_layout()
save_fig(fig, DIR_PLOTS, 'RES-03_heatmap.png',
    'Accuracy heatmap: class × fraction for both experiments')
plt.show()

In [ ]:
# FINAL SUMMARY TABLE

for exp_id, exp_cfg in EXPERIMENTS.items():
    print(f'\n{"="*70}')
    print(f'  {exp_cfg["label"]}')
    print(f'{"="*70}')
    print(f'  {"Fraction":>10}  {"Granite":>9}  {"Sandstone":>10}  {"Limestone":>10}  {"Overall":>9}')
    print(f'  {"-"*60}')
    for fraction in FRACTIONS:
        if fraction not in agg[exp_id]: continue
        d = agg[exp_id][fraction]
        print(f'  {int(fraction*100):>9}%'
              f'  {d[0]["mean"]:>7.1f}%'
              f'  {d[1]["mean"]:>9.1f}%'
              f'  {d[2]["mean"]:>9.1f}%'
              f'  {d["overall"]["mean"]:>8.1f}%')
    print(f'{"="*70}')

print('\nKey interpretation:')
print('  If accuracy is high at low fractions → speed generalisation is strong')
print('  If accuracy grows sharply with fraction → more data helps cross-speed')
print('  If Exp A ≈ Exp B → speed direction does not matter (symmetric)')
print('  If Exp A ≠ Exp B → one direction generalises better than the other')

In [ ]:
index_path = os.path.join(RESULTS_DIR, 'RESULTS_INDEX.txt')
with open(index_path, 'w') as f:
    f.write('RESULTS INDEX — cross_speed_generalisation\n')
    f.write('=' * 70 + '\n')
    f.write('Exp A: train new 5.10Hz fractions, test new 1.83Hz\n')
    f.write('Exp B: train new 1.83Hz fractions, test new 5.10Hz\n')
    f.write(f'FRACTIONS={[int(f*100) for f in FRACTIONS]}\n')
    f.write(f'SEEDS={SEEDS}\n')
    f.write(f'EPOCHS={EPOCHS}  LR={LR}  WD={WD}  BATCH={BATCH_SIZE}\n\n')
    f.write('FILES\n' + '-'*70 + '\n')
    for path, desc in _saved_files:
        f.write(f'  {os.path.basename(path)}\n    {desc}\n\n')
_saved_files.append((index_path, 'Results index'))
print(f'[SAVED] {index_path}')
print(f'Total files: {len(_saved_files)}')